### pre process data, embed data, and try to retrieve something

### Import Dependencies

In [ ]:
import pandas as pd
import openai

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

In [ ]:
df_items = pd.read_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", orient="records", lines=True)
print(len(df_items))
df_items.head()

In [ ]:
list(df_items["features"].items())[0]

In [ ]:
list(df_items["images"].items())[0]

### pre process items and features

In [ ]:
def preprocess_description(row):
    return f"{row['title']}: {' '.join(row['features'])}"

In [ ]:
def extract_first_large_image(row):
    return row['images'][0].get("large", "")

In [ ]:
df_items["preprocessed_description"] = df_items.apply(preprocess_description, axis=1)
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)
df_items.head()

In [ ]:
# list(df_items["preprocessed_description"].items())[0]
list(df_items["image"].items())[0]

### Sample 50 items from the dataset

In [ ]:
df_items_sample = df_items.sample(n=50, random_state=42) 
len(df_items_sample)

In [ ]:
data_to_embed = df_items_sample[["preprocessed_description", "image", "rating_number", "price",  "average_rating", "parent_asin"]]

In [ ]:
data_to_embed.head()

In [ ]:
# transform to a list of dictionaries whre coluimns represent the keys in the dictionaries
data_to_embed = data_to_embed.to_dict(orient="records")


In [ ]:
data_to_embed

In [ ]:
# we will use openai to embed the data
# define the embedding function

client = openai.OpenAI()

response = client.embeddings.create(
    input="Random text",
    model="text-embedding-3-small"
)


In [ ]:
response.data

In [ ]:
def get_embedding(text):
    response = client.embeddings.create(
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

In [ ]:
get_embedding("Random text")


### CREATE QUADRANT EMBEDDING

In [ ]:
 qdrant_client = QdrantClient(url="http://localhost:6333")
 

In [ ]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-01",
    vectors_config=VectorParams(
        size=1536,
        distance=Distance.COSINE
    )
)

#### Embed data

In [ ]:
pointstruct = PointStruct(
    id=0, 
    vector=get_embedding("Hello world"), 
    payload={"text": "Hello world!", "model": "text-embedding-3-small"}
)
pointstruct

In [ ]:
# future examples we will be doing batch embedding .... 
pointstructs = []
for i, data in enumerate(data_to_embed):
    embedding = get_embedding(data['preprocessed_description'])
    pointstruct = PointStruct(
        id=i,
        vector=embedding,
        payload=data
    ) 
    pointstructs.append(pointstruct)

len(pointstructs)

In [ ]:
pointstructs[0]

### Write embedded data to qdrant

In [ ]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-01",
    points=pointstructs
)

In [ ]:
# if we upsert that means an id collision would be taken into account and new values would be. written to the collection

### Define a function for data retrieval

In [ ]:
def retrieve_data(query, k=5): # 5 most similar items to users query
    embedding = get_embedding(query) # so we are actually creating related vector here
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=embedding,
        limit=k
    )
    return results

In [ ]:
retrieve_data("Do you have a USB connectable fan for hot summers.", k=10).points
